# LipiLens - Colab inference server

Click **Run all**. About 5 minutes, most of it downloading the model.

Before you start: `Runtime` -> `Change runtime type` -> **T4 GPU** -> `Save`.

Nothing to upload - the notebook fetches the server file itself.

Keep this tab open and awake. If the server dies mid-session, **re-run cell 5**
to bring it straight back.

In [ ]:
# ---- 1. dependencies (pinned to match the local environment) ----
!pip install -q "transformers==5.17.0" "peft==0.21.0" \
              "bitsandbytes==0.50.2" "accelerate==1.15.0" \
              pillow fastapi uvicorn pyngrok requests

In [ ]:
# ---- 2. confirm the GPU, and record which one (the paper needs this) ----
import json
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > pick T4 GPU > Save, then reconnect.")

GPU_FACTS = {
    "device_name": torch.cuda.get_device_name(0),
    "vram_gb": round(torch.cuda.get_device_info(0).total_memory / 1024**3, 1),
    "torch": torch.__version__,
}
print(json.dumps(GPU_FACTS, indent=2))

In [ ]:
# ---- 3. fetch the server file (no manual upload needed) ----
import hashlib
import os
import urllib.request

URL = ("https://raw.githubusercontent.com/ckhot18/lipilensv2/"
       "main/colab/lipilens_colab_server.py")
DST = "lipilens_colab_server.py"

def have_server():
    return os.path.exists(DST) and os.path.getsize(DST) > 1000

try:
    data = urllib.request.urlopen(URL, timeout=30).read()
    if b"FastAPI" not in data:
        raise ValueError("downloaded file is not the server")
    with open(DST, "wb") as f:
        f.write(data)
    print(f"fetched {len(data)} bytes  sha256={hashlib.sha256(data).hexdigest()[:12]}")
except Exception as exc:
    print("auto-fetch failed:", type(exc).__name__, exc)

if not have_server():
    print("\nUpload it instead: folder icon (left panel) > upload icon >\n          colab/lipilens_colab_server.py, then re-run this cell.")
    raise SystemExit("server file missing")

print("ready:", os.path.abspath(DST))

In [ ]:
# ---- 4. start the server ----
# Runs it as a subprocess writing to server.log, so a crash is readable
# instead of vanishing into a dead thread. Safe to re-run any time.
import os
import subprocess
import sys
import time

import requests

DST = globals().get("DST", "lipilens_colab_server.py")
PORT, LOG, PIDF = 8000, "server.log", "server.pid"
PROC = None  # the live child, so we can stop it portably

if not (os.path.exists(DST) and os.path.getsize(DST) > 1000):
    raise SystemExit(f"server file missing: {DST} - run cell 3 (or upload it) first")


def healthy():
    try:
        return requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3).ok
    except Exception:
        return False


def stop_previous():
    global PROC
    # Popen.kill() works everywhere; os.kill(pid, signal.SIGKILL) raises
    # AttributeError on platforms where SIGKILL is not defined.
    if PROC is not None and PROC.poll() is None:
        try:
            PROC.kill()
            PROC.wait(timeout=10)
        except Exception:
            pass
    PROC = None
    for path in (PIDF,):
        try:
            os.remove(path)
        except OSError:
            pass
    # then clear anything else holding the port
    try:
        subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True, timeout=10)
    except Exception:
        pass


def start():
    global PROC
    stop_previous()
    time.sleep(1)
    handle = open(LOG, "w")
    proc = subprocess.Popen(
        [sys.executable, "-u", DST], stdout=handle, stderr=subprocess.STDOUT
    )
    PROC = proc
    with open(PIDF, "w") as f:
        f.write(str(proc.pid))
    print(f"started pid {proc.pid}, waiting for /health ...")
    deadline = time.time() + 90
    while time.time() < deadline:
        # Death is checked FIRST: otherwise an unrelated server already on the
        # port would answer /health and hide our crash.
        if proc.poll() is not None:
            print(f"\nSERVER EXITED (code {proc.returncode}). Log tail:\n")
            print(open(LOG).read()[-3000:])
            return False
        if healthy():
            print("health:", requests.get(f"http://127.0.0.1:{PORT}/health").json())
            return True
        time.sleep(2)
    print("\nTIMED OUT after 90s. Log tail:\n")
    print(open(LOG).read()[-3000:])
    return False


if healthy():
    print("already running - reusing it")
    SERVER_UP = True
else:
    SERVER_UP = start()
assert SERVER_UP, "server did not start - read the log above"


## Expose port 8000 to your laptop

**Either** run the next cell for an ngrok URL (needs a free authtoken from
<https://dashboard.ngrok.com/get-started/your-authtoken>),

**or** run the cell after it for Colab's built-in tunnel, which needs no signup.

In [ ]:
# ---- 5a. tunnel via ngrok ----
# The token is looked up in four places so it never has to live in this file.
# Option A (edit PASTE_TOKEN once) also saves it to ngrok_token.txt, which
# lives only inside this Colab runtime and survives runtime restarts.
import os

from pyngrok import ngrok

PASTE_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
TOKEN_FILE = "ngrok_token.txt"


def find_token():
    try:
        from google.colab import userdata

        secret = userdata.get("NGROK_AUTHTOKEN")
        if secret:
            print("token: from Colab secret")
            return secret
    except Exception:
        pass
    env = os.environ.get("NGROK_AUTHTOKEN")
    if env:
        print("token: from environment variable")
        return env
    if os.path.exists(TOKEN_FILE):
        saved = open(TOKEN_FILE).read().strip()
        if saved:
            print("token: saved by an earlier run of this cell")
            return saved
    return PASTE_TOKEN


TOKEN = find_token()

if TOKEN.startswith("PASTE"):
    TUNNEL_URL = None
    print("")
    print("  NO TOKEN FOUND - pick one of these:")
    print("")
    print("  A) edit PASTE_TOKEN at the top of this cell, paste your authtoken")
    print("     from https://dashboard.ngrok.com/get-started/your-authtoken,")
    print("     then run this cell again. It gets saved to ngrok_token.txt so you")
    print("     only ever do this once.")
    print("")
    print("  B) click the key icon in the Colab sidebar, add a secret named")
    print("     NGROK_AUTHTOKEN, tick notebook access, then run this cell again.")
    print("")
    print("  C) no signup at all: skip this and run cell 5b instead.")
    print("")
else:
    if not os.path.exists(TOKEN_FILE):
        try:
            with open(TOKEN_FILE, "w") as f:
                f.write(TOKEN)
            print("saved to ngrok_token.txt - you will not need to paste again")
        except Exception:
            pass
    ngrok.set_auth_token(TOKEN)
    TUNNEL_URL = ngrok.connect(8000).public_url
    print("COLAB_ENDPOINT_URL=" + TUNNEL_URL)


In [ ]:
# ---- 5b. tunnel via Colab (no signup) ----
# Run THIS instead of 5a if you do not want an ngrok account.
from google.colab import output

output.serve_kernel_port_as_window(8000, path="/health")
TUNNEL_URL = "PASTE_THE_URL_SHOWN_IN_THE_POPUP"

In [ ]:
# ---- 6. self-test through the tunnel ----
# Loads the model (1-3 min) and transcribes one line. Only trust the URL
# printed above once this returns 200.
import base64
import time
import io

from PIL import Image, ImageDraw

if not TUNNEL_URL or TUNNEL_URL.startswith("PASTE"):
    raise SystemExit(
        "No tunnel URL yet.\n"
        "  -> run cell 5a and follow the instructions it prints (it needs your\n"
        "     ngrok authtoken), OR run cell 5b for Colab's built-in tunnel.\n"
        "  -> then re-run this cell."
    )

print("remote GPU facts:", requests.get(TUNNEL_URL + "/gpu", timeout=30).json())

img = Image.new("RGB", (900, 90), "white")
ImageDraw.Draw(img).text((10, 32), "test line 12345", fill="black")
buf = io.BytesIO()
img.save(buf, format="PNG")

t0 = time.time()
resp = requests.post(
    TUNNEL_URL + "/transcribe",
    json={
        "image": base64.b64encode(buf.getvalue()).decode(),
        "prompt": ("This image contains handwritten text in Modi script, a "
                   "historical cursive script used to write the Marathi "
                   "language. Transliterate the text in this image into "
                   "Devanagari script. Output only the Devanagari text, with "
                   "no explanation."),
    },
    timeout=900,
)
print(f"\nHTTP {resp.status_code} in {time.time() - t0:.0f}s")
print(resp.text[:600])
print("\nSERVER IS WORKING - copy the URL into your .env"
      if resp.status_code == 200 else "\nFAILED - read the output above")

## On your own machine

Edit `.env` in the project root:

```
INFERENCE_MODE=colab
COLAB_ENDPOINT_URL=<the URL printed in cell 5>
```

```bash
python -m uvicorn backend.main:app --port 8000
cd frontend && npm run dev
```

Check <http://localhost:8000/api/health> - it must report `"status": "ok"`.
If it says `degraded`, the URL is wrong or the tunnel dropped.

**If the server dies mid-session:** re-run cell 4, then cell 5 again (ngrok
gives you a new URL each time, so update `.env` again).